# Klasyfikacja komórek krwi obwodowej za pomocą CNN
**Dataset:** Blood Cells Image Dataset (PBC) — Acevedo et al., 2020  
**Zadanie:** Klasyfikacja obrazów mikroskopowych do 8 klas komórek  
**Środowisko:** Google Colab + GPU (T4)  

**Klasy:** neutrofile, eozynofile, bazofile, limfocyty, monocyty, niedojrzałe granulocyty, erytroblasty, płytki krwi

---
## Pobieranie danych z Kaggle

In [4]:
# Instalacja kagglehub
!pip install kagglehub -q

import kagglehub

# Pobieranie datasetu
path = kagglehub.dataset_download("unclesamulus/blood-cells-image-dataset")
print(f"Dataset jest pod: {path}")

100%|██████████| 268M/268M [00:16<00:00, 16.7MB/s]

Extracting files...


Dataset jest pod: /root/.cache/kagglehub/datasets/unclesamulus/blood-cells-image-dataset/versions/2


In [5]:
import os

path = kagglehub.dataset_download("unclesamulus/blood-cells-image-dataset")
DATA_DIR = path
print(f"Dataset: {DATA_DIR}")

# Struktura katalogow
for root, dirs, files_list in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 2:
            print(f'{indent}  ... ({len(files_list)} plikow)')

# Automatycznie znajdz katalog z klasami
CLASSES_DIR = None
for root, dirs, files_list in os.walk(DATA_DIR):
    if len(dirs) >= 5 and not any(d.startswith('.') for d in dirs):
        CLASSES_DIR = root
        break

print(f"Katalog z klasami: {CLASSES_DIR}")
print(f"Klasy: {sorted(os.listdir(CLASSES_DIR))}")

Using Colab cache for faster access to the 'blood-cells-image-dataset' dataset.
Dataset: /kaggle/input/blood-cells-image-dataset
blood-cells-image-dataset/
  bloodcells_dataset/
    monocyte/
      ... (1420 plikow)
    ig/
      ... (2895 plikow)
    neutrophil/
      ... (3329 plikow)
    basophil/
      ... (1218 plikow)
    lymphocyte/
      ... (1214 plikow)
    erythroblast/
      ... (1551 plikow)
    eosinophil/
      ... (3117 plikow)
    platelet/
      ... (2348 plikow)
Katalog z klasami: /kaggle/input/blood-cells-image-dataset/bloodcells_dataset
Klasy: ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']


---
## Sekcja 1: Importy i ustawienia

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

torch.manual_seed(1)  # dla powtarzalnosci wynikow

# Uzyj GPU jesli dostepne
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Urzadzenie: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Sekcja 2: Eksploracja danych (EDA)

In [ ]:
# Policz obrazy w kazdej klasie
class_names = sorted(os.listdir(CLASSES_DIR))
class_counts = {}
for cls in class_names:
    cls_path = os.path.join(CLASSES_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        class_counts[cls] = count

print("Rozklad klas:")
for cls, count in class_counts.items():
    print(f"  {cls:40s}: {count} obrazow")
print(f"\nLacznie: {sum(class_counts.values())} obrazow")

In [ ]:
# Histogram rozkladu klas
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(), color='steelblue', edgecolor='navy')
ax.set_title('Rozklad klas w datasecie PBC', fontsize=14)
ax.set_xlabel('Typ komorki')
ax.set_ylabel('Liczba obrazow')
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(val), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Przykladowe obrazy z kazdej klasy (analogicznie do CIFAR10 z notebooka kursowego)
from PIL import Image

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 8))

for cls, ax in zip(class_counts.keys(), axes.flatten()):
    cls_path = os.path.join(CLASSES_DIR, cls)
    img_file = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png'))][0]
    img = Image.open(os.path.join(cls_path, img_file))
    ax.imshow(img)
    ax.set_title(cls, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Przykladowe obrazy z kazdej klasy', fontsize=14)
plt.tight_layout()
plt.show()

---
## Sekcja 3: Dataset i DataLoader

In [ ]:
# Parametry
IMG_SIZE    = 128   # oryginalne: 360x363, zmniejszamy dla szybszego trenowania
BATCH_SIZE  = 64
NUM_CLASSES = 8

# Transformacje dla zbioru treningowego (z augmentacja)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),              # losowe odbicie poziome
    transforms.RandomRotation(15),                  # losowy obrot +/-15 stopni
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # zmiana jasnosci/kontrastu
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # normalizacja do [-1, 1]
])

# Transformacje dla walidacji i testu (bez augmentacji!)
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Transformacje zdefiniowane.")

In [ ]:
# Wczytanie calego datasetu przez ImageFolder
# ImageFolder automatycznie przypisuje etykiety na podstawie nazw folderow
full_dataset_train = datasets.ImageFolder(root=CLASSES_DIR, transform=train_transform)
full_dataset_eval  = datasets.ImageFolder(root=CLASSES_DIR, transform=val_test_transform)

print(f"Lacznie: {len(full_dataset_train)} obrazow")
print(f"Klasy: {full_dataset_train.classes}")
print(f"Ksztalt obrazu: {full_dataset_train[0][0].shape}")

In [ ]:
# Podzial na train/val/test (70% / 15% / 15%)
total    = len(full_dataset_train)
n_train  = int(0.70 * total)
n_val    = int(0.15 * total)
n_test   = total - n_train - n_val

# Generujemy indeksy podziau
from torch.utils.data import Subset
indices = torch.randperm(total).tolist()
train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

# Train korzysta z augmentacji, val/test nie
train_dataset = Subset(full_dataset_train, train_idx)
val_dataset   = Subset(full_dataset_eval,  val_idx)
test_dataset  = Subset(full_dataset_eval,  test_idx)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# DataLoadery
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Liczba batchy: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

---
## Sekcja 4: Funkcje pomocnicze (accuracy + petla treningowa)

In [ ]:
def get_accuracy(model, loader):
    """Oblicza dokladnosc modelu na danym zbiorze."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)       # gdzie najwieksza wartosc
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def train(model, train_loader, val_loader, num_epochs=20, learn_rate=0.001):
    """Petla treningowa — zwraca historie loss i accuracy."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learn_rate)

    losses, train_acc, val_acc = [], [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            # Forward
            out = model(imgs)
            loss = criterion(out, labels)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Sredni loss z epoki
        epoch_loss = running_loss / len(train_loader)
        losses.append(epoch_loss)

        # Accuracy raz na epoke
        t_acc = get_accuracy(model, train_loader)
        v_acc = get_accuracy(model, val_loader)
        train_acc.append(t_acc)
        val_acc.append(v_acc)

        print(f"Epoka [{epoch+1:02d}/{num_epochs}]  Loss: {epoch_loss:.4f}  "
              f"Train Acc: {t_acc*100:.1f}%  Val Acc: {v_acc*100:.1f}%")

    return losses, train_acc, val_acc


def plot_history(losses, train_acc, val_acc, title=''):
    """Rysuje wykresy loss i accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(losses, color='tomato')
    ax1.set_title(f'{title} - Loss (train)')
    ax1.set_xlabel('Epoka')
    ax1.set_ylabel('CrossEntropyLoss')

    ax2.plot(train_acc, label='Train', color='steelblue')
    ax2.plot(val_acc,   label='Val',   color='orange')
    ax2.set_title(f'{title} - Accuracy')
    ax2.set_xlabel('Epoka')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


print("Funkcje pomocnicze gotowe.")

---
## Sekcja 5: Model 1 — Wlasna siec CNN (BloodCellNet)

Architektura oparta na ConvNet z notebooka kursowego, rozbudowana do 3 blokow konwolucyjnych dla kolorowych obrazow RGB.

In [ ]:
class BloodCellNet(nn.Module):
    def __init__(self, num_classes=8):
        super(BloodCellNet, self).__init__()

        # Blok 1: 3 kanaly RGB -> 32 filtry; 128x128 -> 64x64
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Blok 2: 32 -> 64 filtry; 64x64 -> 32x32
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Blok 3: 64 -> 128 filtry; 32x32 -> 16x16
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.dropout = nn.Dropout(0.5)

        # Po 3x MaxPool: 128 / 2 / 2 / 2 = 16 -> 128 * 16 * 16 = 32768
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.reshape(out.size(0), -1)   # splaszczenie
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return out


model_cnn = BloodCellNet(num_classes=NUM_CLASSES).to(device)

# Liczba parametrow do trenowania
total_params = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"BloodCellNet — liczba parametrow: {total_params:,}")

In [ ]:
# Trenowanie wlasnej sieci CNN
print("=== Trening BloodCellNet ===")
NUM_EPOCHS_CNN = 20

losses_cnn, train_acc_cnn, val_acc_cnn = train(
    model_cnn, train_loader, val_loader,
    num_epochs=NUM_EPOCHS_CNN, learn_rate=0.001
)

In [ ]:
plot_history(losses_cnn, train_acc_cnn, val_acc_cnn, title='BloodCellNet (wlasna CNN)')

In [ ]:
test_acc_cnn = get_accuracy(model_cnn, test_loader)
print(f"BloodCellNet — Accuracy na zbiorze testowym: {test_acc_cnn*100:.2f}%")

---
## Sekcja 6: Model 2 — Transfer Learning (ResNet-18)

Uzywamy modelu wytrenowanego na ImageNet (1.2M obrazow, 1000 klas). Podmieniamy tylko ostatnia warstwe na nasza klasyfikacje 8 klas — siec "zna juz" ogolne cechy wizualne (krawedzie, tekstury).

In [ ]:
# Zaladuj ResNet-18 z wagami ImageNet
model_resnet = models.resnet18(pretrained=True)

# Zamroz wagi backbone (nie trenujemy warstw konwolucyjnych na poczatek)
for param in model_resnet.parameters():
    param.requires_grad = False

# Podmien ostatnia warstwe (fc) na nasz klasyfikator 8 klas
num_features = model_resnet.fc.in_features   # 512 dla ResNet-18
model_resnet.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES)
)

model_resnet = model_resnet.to(device)

trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
print(f"ResNet-18 — parametry do trenowania (tylko glowa): {trainable:,}")

In [ ]:
# Faza 1: trenuj tylko nowa glowe (kilka epok)
print("=== Transfer Learning — Faza 1: trenowanie glowy ===")
losses_r1, train_acc_r1, val_acc_r1 = train(
    model_resnet, train_loader, val_loader, num_epochs=5, learn_rate=0.001
)

In [ ]:
# Faza 2: odmroz caly model i finetunuj z malym LR
for param in model_resnet.parameters():
    param.requires_grad = True

print("=== Transfer Learning — Faza 2: fine-tuning calosci (mniejszy LR) ===")
losses_r2, train_acc_r2, val_acc_r2 = train(
    model_resnet, train_loader, val_loader, num_epochs=10, learn_rate=0.0001
)

In [ ]:
plot_history(losses_r2, train_acc_r2, val_acc_r2, title='ResNet-18 (fine-tuning)')

In [ ]:
test_acc_resnet = get_accuracy(model_resnet, test_loader)
print(f"ResNet-18 — Accuracy na zbiorze testowym: {test_acc_resnet*100:.2f}%")

---
## Sekcja 7: Porownanie modeli

In [ ]:
print("=" * 50)
print(f"{'Model':<25} {'Test Accuracy':>15}")
print("=" * 50)
print(f"{'BloodCellNet (wlasna)':<25} {test_acc_cnn*100:>14.2f}%")
print(f"{'ResNet-18 (TL)':<25} {test_acc_resnet*100:>14.2f}%")
print("=" * 50)

---
## Sekcja 8: ⭐ Macierz pomylek (Confusion Matrix)

Macierz pomylek pokazuje ktore klasy komorek sa ze soba mylone.  
W diagnostyce medycznej to kluczowa informacja — np. czy model myli bazofile z eozynofilami?

In [ ]:
def get_predictions(model, loader):
    """Zbiera prawdziwe etykiety i predykcje modelu."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def plot_confusion_matrix(model, loader, class_names, title='Confusion Matrix'):
    """Rysuje macierz pomylek jako heatmape (znormalizowana wierszami)."""
    y_true, y_pred = get_predictions(model, loader)
    cm = confusion_matrix(y_true, y_pred)

    # Normalizacja: kazdy wiersz sumuje sie do 1 (procent poprawnych dla kazdej klasy)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Predykcja')
    ax.set_ylabel('Prawdziwa klasa')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    return y_true, y_pred


print("Funkcje ewaluacji gotowe.")

In [ ]:
class_labels = full_dataset_train.classes

print("--- Macierz pomylek: BloodCellNet ---")
y_true_cnn, y_pred_cnn = plot_confusion_matrix(
    model_cnn, test_loader, class_labels,
    title='BloodCellNet - Confusion Matrix (zbior testowy)'
)

In [ ]:
print("--- Macierz pomylek: ResNet-18 ---")
y_true_rn, y_pred_rn = plot_confusion_matrix(
    model_resnet, test_loader, class_labels,
    title='ResNet-18 - Confusion Matrix (zbior testowy)'
)

In [ ]:
# Szczegolowy raport: precision, recall, F1 dla kazdej klasy
print("=== Raport klasyfikacji — ResNet-18 ===")
print(classification_report(y_true_rn, y_pred_rn, target_names=class_labels))

---
## Sekcja 9: ⭐ Wizualizacja blednych predykcji

Wyswietlamy obrazy, ktore model sklasyfikowal zle — pomaga zrozumiec gdzie i dlaczego model sie myli.

In [ ]:
def show_mistakes(model, loader, class_names, n=10, title='Bledne predykcje'):
    """Wyswietla n obrazow, ktore model sklasyfikowal blednie."""
    model.eval()
    mistakes = []  # lista (obraz, prawdziwa_klasa, predykowana_klasa)

    with torch.no_grad():
        for imgs, labels in loader:
            imgs_dev = imgs.to(device)
            outputs  = model(imgs_dev)
            preds    = outputs.argmax(dim=1).cpu()

            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    mistakes.append((imgs[i], labels[i].item(), preds[i].item()))
                if len(mistakes) >= n:
                    break
            if len(mistakes) >= n:
                break

    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    fig.suptitle(title, fontsize=13)

    for ax, (img, true, pred) in zip(axes.flatten(), mistakes):
        # Cofnij normalizacje: img * 0.5 + 0.5 (analogicznie jak w notebooku kursowym)
        img_show = img * 0.5 + 0.5
        img_show = img_show.permute(1, 2, 0)  # CxHxW -> HxWxC dla matplotlib
        img_show = img_show.clamp(0, 1)
        ax.imshow(img_show)
        ax.set_title(f"Prawda: {class_names[true]}\nPred: {class_names[pred]}",
                     fontsize=8, color='red')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()


show_mistakes(model_resnet, test_loader, class_labels, n=10,
              title='ResNet-18 - przykladowe bledne predykcje')

---
## Sekcja 10: Zapis modeli na Google Drive

In [ ]:
# Zapisz wytrenowane modele — przezyja reset sesji Colab!
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/blood_cells_project'
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model_cnn.state_dict(),    f'{SAVE_DIR}/blood_cell_net.pth')
torch.save(model_resnet.state_dict(), f'{SAVE_DIR}/resnet18_finetuned.pth')

print(f"Modele zapisane w: {SAVE_DIR}")
print("Aby wczytac pozniej: model.load_state_dict(torch.load('...pth'))")

---
## Podsumowanie wynikow

| Model | Architektura | Parametry | Test Accuracy |
|---|---|---|---|
| BloodCellNet | 3x Conv+BN+ReLU+MaxPool + FC | ~4M | uzupelnij |
| ResNet-18 (TL) | 18 warstw + fine-tuning | ~11M | uzupelnij |

**Co warto omowic w raporcie:**
- Wplyw augmentacji na generalizacje (komorki moga byc sfotografowane pod roznym katem)
- Transfer learning daje wynik przy mniejszej liczbie epok  
- Macierz pomylek ujawnia trudne do rozroznienia pary klas (np. bazofile vs eozynofile)
- Zbior jest lekko niezbalansowany (neutrofile 3x wiecej niz limfocyty)